# PINNs for Schwarzschild orbits: the orbit equation is the loss (local CPU)

Equatorial orbits around a non-rotating black hole, in Binet form ($u = M/r$ as a
function of the orbital angle $\\varphi$):

- light: $u'' + u = 3u^2$
- a massive body: $u'' + u = 1/p + 3f\\,u^2$, where $f = 1$ is Einstein and $f = 0$ is Newton

Three experiments, all physics-informed (the networks learn from the equation's
residual, not from solved orbits):

1. **One ray, from the equation alone.** How close to the photon sphere
   ($b \\to b_\\text{crit} = 3\\sqrt3\\,M$) can a PINN get the deflection right?
2. **Every ray at once.** One network $u(\\varphi, b)$ for $b \\in [5.25, 30]$, a
   deflection surrogate trained on no data.
3. **Weighing relativity from an orbit.** From a few noisy stellar positions, recover
   $p$ and $f$: can it tell Einstein from Newton, versus the classical shooting fit?

This is the **local** version of the Kaggle notebook: it imports the engine from this
checkout instead of cloning, writes results next to itself, and uses every core on the
machine (one tiny network per core, many at once). It was about 45 minutes on Kaggle's
2 cores; on 12 it is bounded by the longest single job, so expect a handful of minutes.
No accelerator is needed (CPU only).

In [ ]:
# Local run: import the engine from this checkout (no git clone, no /kaggle paths).
import os
import json
import math
import time

import numpy as np
import torch
from IPython.display import Image, display

try:
    from bhml import pinn as P              # repo installed with `pip install -e .`
except ModuleNotFoundError:
    # not installed: add the ml/ dir (this notebook lives in ml/notebooks/) to the path.
    # Note: on Windows the worker processes use spawn and re-import bhml, so an editable
    # install (pip install -e .) is the reliable way to let them find it.
    import sys
    here = os.getcwd()
    for up in (".", "..", "../..", "ml", "../ml"):
        cand = os.path.abspath(os.path.join(here, up))
        if os.path.isdir(os.path.join(cand, "bhml")):
            sys.path.insert(0, cand)
            break
    from bhml import pinn as P

WORKERS = os.cpu_count()
OUT = os.path.abspath("pinn_results")
os.makedirs(OUT, exist_ok=True)
print("cores:", WORKERS, "| torch", torch.__version__, "| out:", OUT)

## Settings

In [ ]:
FORWARD = dict(P.FORWARD)    # tanh MLP 32x3, Adam 3000 + L-BFGS 1000 per window, 1 rad windows
PARAM = dict(P.PARAM)        # u(phi, b): tanh MLP 64x4, Adam 10000 + L-BFGS 2000, b in [5.25, 30]
INVERSE = dict(P.INVERSE)    # u(phi): tanh MLP 64x4, Adam 3000 + L-BFGS 1000, p and f trainable

GAPS = [10, 3, 1, 0.3, 0.1, 0.03, 0.01, 1e-3, 1e-4]   # b - b_crit for the single-ray study
ENCODINGS = ["log", "lin"]                            # how the parametric network sees b
NOISES = [0.001, 0.003, 0.01, 0.03, 0.1]              # noise std / mean u
N_OBS = [10, 30, 100]                                  # observed positions (at 1% noise)
N_ORBITS = [1, 2, 3]                                   # turns observed (at 1% noise, 30 points)
SEEDS = [0, 1, 2, 3]                                   # noise draws per setting
ORBIT = dict(p=20.0, e=0.5)                            # periapsis 13.3 M, apoapsis 40 M

## Run everything

All jobs go into one pool, the longest first, one core each.

In [ ]:
jobs = [(P.param_job, dict(cfg=dict(PARAM, enc=e))) for e in ENCODINGS]
jobs += [(P.forward_job, dict(b=P.B_CRIT + g, mode=m, cfg=FORWARD))
         for g in GAPS for m in ("march", "single")]
inv = [dict(ORBIT, f=f, noise=n, seed=s) for f in (1.0, 0.0) for n in NOISES for s in SEEDS]
inv += [dict(ORBIT, f=1.0, noise=0.01, n_obs=k, seed=s) for k in N_OBS if k != 30 for s in SEEDS]
inv += [dict(ORBIT, f=1.0, noise=0.01, n_orbits=k, seed=s) for k in N_ORBITS if k != 3 for s in SEEDS]
jobs += [(P.inverse_job, dict(kw, cfg=INVERSE)) for kw in inv]
print(len(jobs), "jobs on", WORKERS, "cores")
t0 = time.time()
res = P.run_parallel(P.call, jobs, WORKERS, label="jobs")
print(f"all done in {(time.time() - t0) / 60:.1f} min")
param_res = dict(zip(ENCODINGS, res[:len(ENCODINGS)]))
fwd_res = res[len(ENCODINGS):len(ENCODINGS) + 2 * len(GAPS)]
inv_res = res[len(ENCODINGS) + 2 * len(GAPS):]

## 1. One ray: how close to the photon sphere

Relative error of the deflection, with the turning point read off where $u'$ first
turns negative. "single" is one network over a domain just long enough to hold the
turning point (it is given the answer's length); "march" goes one radian at a time
from where the last window ended and needs nothing.

In [ ]:
print(f"{'b - b_crit':>10s} {'exact':>8s} | {'single: error':>13s} {'time':>6s} | {'march: error':>12s} {'windows':>7s} {'time':>6s}")
for g in GAPS:
    s = next(r for r in fwd_res if r["mode"] == "single" and abs(r["b"] - P.B_CRIT - g) < 1e-12)
    m = next(r for r in fwd_res if r["mode"] == "march" and abs(r["b"] - P.B_CRIT - g) < 1e-12)
    print(f"{g:10.0e} {m['exact']:8.3f} | {s['rel_err']:13.2e} {s['time']:5.0f}s | {m['rel_err']:12.2e} {m['n_windows']:7d} {m['time']:5.0f}s")
P.plot_forward(fwd_res, f"{OUT}/pinn_forward.png")
display(Image(f"{OUT}/pinn_forward.png"))

## 2. Every ray at once: u(phi, b)

One network, physics only, evaluated on 200 values of $b$ against the exact deflection.

In [ ]:
for name, r in param_res.items():
    print(f"{name}: relative L2 {100 * r['rel_l2']:.2f}%, worst {100 * r['max_rel']:.2f}%, "
          f"no turning point for {r['n_nan']} b, final loss {r['loss']:.2e}, trained in {r['train_time'] / 60:.1f} min")
    b, pred, ex = map(np.array, (r["b"], r["pred"], r["exact"]))
    for lo, hi in ((P.B_CRIT, 5.5), (5.5, 7), (7, 12), (12, 30.01)):
        s = (b >= lo) & (b < hi)
        print(f"   b in [{lo:.2f}, {hi:g}): mean error {100 * np.nanmean(abs(pred[s] - ex[s]) / ex[s]):.2f}%")
P.plot_param(param_res, f"{OUT}/pinn_parametric.png")
display(Image(f"{OUT}/pinn_parametric.png"))

## 3. Weighing relativity from an orbit

Orbit with $p = 20$, $e = 0.5$, observed at random angles over its turns, Gaussian
noise on $u$. The PINN learns the orbit and $(p, f)$ together; the shooting fit
integrates the ODE and fits its initial conditions, $p$ and $f$ by least squares.
Mean and spread over the noise draws.

In [ ]:
def show(rows, by):
    print(f"{by:>9s} {'f true':>6s} | {'PINN f':>15s} {'p err':>7s} | {'shooting f':>15s} {'p err':>7s}")
    for r in rows:
        a, s = r["pinn"], r["shooting"]
        print(f"{r[by]:9g} {r['f_true']:6g} | {a['f_mean']:7.3f} ± {a['f_std']:5.3f} {100 * a['p_err']:6.2f}% | "
              f"{s['f_mean']:7.3f} ± {s['f_std']:5.3f} {100 * s['p_err']:6.2f}%")

base = [r for r in inv_res if r["n_obs"] == 30 and r["n_orbits"] == 3]
by_noise = P.summarize_inverse(base, "noise")
by_obs = P.summarize_inverse([r for r in inv_res if r["f"] == 1.0 and r["noise"] == 0.01 and r["n_orbits"] == 3], "n_obs")
by_orbits = P.summarize_inverse([r for r in inv_res if r["f"] == 1.0 and r["noise"] == 0.01 and r["n_obs"] == 30], "n_orbits")
show(by_noise, "noise"); print()
show(by_obs, "n_obs"); print()
show(by_orbits, "n_orbits")
print("\nPINN fit time:", f"{np.mean([r['pinn']['time'] for r in inv_res]):.0f}s;",
      "shooting fit:", f"{np.mean([r['shooting']['time'] for r in inv_res]):.1f}s")
P.plot_inverse(P.make_orbit(**ORBIT, f=1.0, noise=0.01), P.make_orbit(**ORBIT, f=0.0, noise=0.01),
               {"GR orbit": [r for r in by_noise if r["f_true"] == 1.0],
                "Newtonian orbit": [r for r in by_noise if r["f_true"] == 0.0]},
               f"{OUT}/pinn_inverse.png")
display(Image(f"{OUT}/pinn_inverse.png"))

In [ ]:
for name, r in param_res.items():
    torch.save({k: torch.tensor(v) for k, v in r.pop("state").items()}, f"{OUT}/param_{name}.pt")
results = dict(forward=fwd_res, param=param_res, inverse=inv_res,
               summary=dict(noise=by_noise, n_obs=by_obs, n_orbits=by_orbits),
               cfg=dict(forward=FORWARD, param=PARAM, inverse=INVERSE, orbit=ORBIT))
json.dump(results, open(f"{OUT}/results.json", "w"), indent=1)
print("saved to", OUT, ":", sorted(os.listdir(OUT)))